In [4]:
# ===== Colab Setup =====

REPO_URL = "https://github.com/dariuscparker/Self-Improving-Agent-Projects.git"
REPO_DIR = "/content/Self-Improving-Agent-Projects"

import os, sys

# Remove any old copy (optional but safe)
if os.path.exists(REPO_DIR):
    !rm -rf {REPO_DIR}

# Clone fresh repo
!git clone {REPO_URL} {REPO_DIR}

# Add repo to Python path
if REPO_DIR not in sys.path:
    sys.path.append(REPO_DIR)

# Move into HW1 directory
%cd /content/Self-Improving-Agent-Projects/modules/01_test_time_compute

# Verify
!pwd
!ls

shell-init: error retrieving current directory: getcwd: cannot access parent directories: No such file or directory
Cloning into '/content/Self-Improving-Agent-Projects'...
fatal: Unable to read current working directory: No such file or directory
[Errno 2] No such file or directory: '/content/Self-Improving-Agent-Projects/modules/01_test_time_compute'
/content/Self-Improving-Agent-Projects/modules/01_test_time_compute
shell-init: error retrieving current directory: getcwd: cannot access parent directories: No such file or directory
pwd: error retrieving current directory: getcwd: cannot access parent directories: No such file or directory
shell-init: error retrieving current directory: getcwd: cannot access parent directories: No such file or directory


# HW: Test-Time Compute Strategies on GSM8K

This notebook reflects the exercises generated by ChatGPT to review the course material in module one of Stanford's CS239A course. In this notebook, we will explore various test time compute strategies to help improve the results of a small model.

The first module illustrates that 'more compute' is typically associated with better performance. Using strategies to increase prompting can help our end result.

We will use the **GSM8K** dataset and a **lightweight open model** (**Qwen/Qwen2.5-1.5B-Instruct**) with 4-bit quantization. We will show how a lightweight model can achieve improved results with test time compute strategies. We will explore the following techniques in this notebook:

- **Part 1:** Single-pass CoT baseline (System-1)
- **Part 2:** Best-of-N sampling + **majority vote** (self-consistency)
- **Part 3:** Best-of-N + **simple verifier** (Proposer–Verifier using labels)
- **Part 4:** **Difficulty binning** (Easy/Medium/Hard) via empirical pass rate
- **Part 5:** **Sequential revisions** (self-refinement chains)
- **Part 6:** Lightweight **beam/search** over partial reasoning
- **Part 7:** Difficulty-aware **routing policy** (compute-optimal TTC) quick

In [5]:
# REPO_URL = "https://github.com/dariuscparker/Self-Improving-Agent-Projects.git"
# REPO_DIR = "/content/Self-Improving-Agent-Projects"

# import os, sys

# %cd /content

# if not os.path.exists(REPO_DIR):
#     !git clone {REPO_URL} {REPO_DIR}
# else:
#     %cd {REPO_DIR}
#     !git pull

# if REPO_DIR not in sys.path:
#     sys.path.insert(0, REPO_DIR)

# %cd /content/Self-Improving-Agent-Projects/modules/01_test_time_compute
# !pwd
# !ls

/content
Cloning into '/content/Self-Improving-Agent-Projects'...
remote: Enumerating objects: 47, done.
remote: Counting objects: 100% (47/47), done.
remote: Compressing objects: 100% (39/39), done.
remote: Total 47 (delta 12), reused 29 (delta 5), pack-reused 0 (from 0)
Receiving objects: 100% (47/47), 137.23 KiB | 1.91 MiB/s, done.
Resolving deltas: 100% (12/12), done.
/content/Self-Improving-Agent-Projects/modules/01_test_time_compute
/content/Self-Improving-Agent-Projects/modules/01_test_time_compute
hw1-test-time-compute.ipynb


## Part 0 — Setup
Set up the environment, load GSM8K and load a lightweight model. For this assignment, we will use Qwen2.5-1.5B Instruct. More information can be found about the model at the link below.

https://huggingface.co/Qwen/Qwen2.5-1.5B-Instruct



In [6]:
!pip -q install -U bitsandbytes transformers accelerate datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 15.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 113.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 30.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 19.5 MB/s eta 0:00:00


In [7]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from datasets import load_dataset

import torch
import re
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from tqdm import tqdm
from collections import Counter

import time

# Global timer start
NOTEBOOK_START_TIME = time.time()


ValueError: pyarrow.lib.IpcReadOptions size changed, may indicate binary incompatibility. Expected 112 from C header, got 104 from PyObject

### Shared Helper Utilities

To keep this notebook readable and modular, reusable utility functions are stored in:

`shared/helpers.py`

This helper module contains common logic used throughout the notebook, including:

- answer extraction
- example display helpers
- single-generation utilities
- majority voting
- evaluation logic for both the baseline and self-consistency settings

In [8]:
# ## Shared Helper Functions
import sys
from pathlib import Path
import importlib

# Locate repo root by finding shared/helpers.py
_repo_root = next(
    (p for p in [Path.cwd(), *Path.cwd().parents] if (p / "shared" / "helpers.py").exists()),
    None
)

if _repo_root is None:
    raise FileNotFoundError("Could not locate shared/helpers.py from the current notebook directory.")

if str(_repo_root) not in sys.path:
    sys.path.insert(0, str(_repo_root))

import shared.helpers as helpers
importlib.reload(helpers)

# Pull commonly used names into notebook scope
final_re = helpers.final_re
extract_true_answer = helpers.extract_true_answer
show_example = helpers.show_example
generate_one = helpers.generate_one
majority_vote = helpers.majority_vote
evaluate = helpers.evaluate

### Load the GSM8K dataset and create train/eval subsets.

In [ ]:
# ## Load and Prepare Dataset
dataset = load_dataset("gsm8k", "main")
print("Dataset loaded successfully!")

# Create subsets (adjust sizes if you want)
train_subset = dataset["train"].select(range(1000))
eval_subset  = dataset["test"].select(range(500))

train_df = train_subset.to_pandas()
eval_df  = eval_subset.to_pandas()

print(f"Training subset: {len(train_df)} examples")
print(f"Evaluation subset: {len(eval_df)} examples")


## Explore the Data

The GSM8K is a collection of middle school math problems that include questions and answers. It is a common benchmark data set for assessing reasoning models.

First, we can convert the dataset to pandas DataFrames for quick structural inspection.

In [ ]:
train_df = pd.DataFrame(dataset["train"])
test_df = pd.DataFrame(dataset["test"])
print("GSM8K train size:", len(train_df))
print("GSM8K test size:", len(test_df))
# Show the first five rows
train_df.head()

We can inspect a few training examples to confirm the question/answer format.

In [ ]:
# Use show_example function from shared/helpers.py to inspect the first three training examples.
for i in range(3):
    show_example(dataset["train"][i])


## Answer Pattern and Regex

It is important to understand the structure of a data set so that a reasoning model can be appropriately prompted. GSM8K solutions end with a standardized final-answer line:

- Final numeric answer appears after `####`
- It is on its own line
- It may be an integer or decimal


Because this format is consistent, we can create a regex pattern to reliably extract the true answer with a regular expression. The regex pattern below is defined in shared/helpers.py:

`final_re = re.compile(r"####\s*([-+]?\d+(?:\.\d+)?)")
`


In [ ]:
# Regex is defined in shared/helpers.py
print("Final answer regex:", final_re.pattern)


Also, notice that each solution not only includes the final answer, but includes reasoning that lead to the final answer. We defined `extract_true_answer` to parse each full solution and keep only the numeric final answer (the value after `####`.)

In [ ]:
dataset = dataset.map(lambda x: {"true_answer": extract_true_answer(x)})


Verify that the extracted `true_answer` aligns with the original `answer` text.


In [ ]:
for i in range(3):
    show_example(dataset["train"][i])



QUESTION:
 Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?

SOLUTION:
 Natalia sold 48/2 = <<48/2=24>>24 clips in May.
Natalia sold 48+24 = <<48+24=72>>72 clips altogether in April and May.
#### 72

TRUE ANSWER: 72

QUESTION:
 Weng earns $12 an hour for babysitting. Yesterday, she just did 50 minutes of babysitting. How much did she earn?

SOLUTION:
 Weng earns 12/60 = $<<12/60=0.2>>0.2 per minute.
Working 50 minutes, she earned 0.2 x 50 = $<<0.2*50=10>>10.
#### 10

TRUE ANSWER: 10

QUESTION:
 Betty is saving money for a new wallet which costs $100. Betty has only half of the money she needs. Her parents decided to give her $15 for that purpose, and her grandparents twice as much as her parents. How much more money does Betty need to buy the wallet?

SOLUTION:
 In the beginning, Betty has only 100 / 2 = $<<100/2=50>>50.
Betty's grandparents gave her 15 * 2 = $<<15*2=30>>30.
This

## Model Setup

In this section, we load a lightweight instruction-tuned language model:

- **Model:** Qwen2.5-1.5B-Instruct  
- **Type:** Causal Language Model (decoder-only transformer)  
- **Use case:** Step-by-step reasoning and instruction following  

This model is intentionally small (~1.5B parameters) so it can run efficiently in a Colab environment while still demonstrating reasoning capabilities. We will use Quantitization which significantly reduces GPU usage without a severe cost to precision.  We will use **4-bit quantization** via the `bitsandbytes` library.


### Check GPU Availability

In [ ]:
# Check GPU Availability
print("CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("Compute Capability:", torch.cuda.get_device_capability(0))


CUDA Available: True
GPU: Tesla T4
Compute Capability: (7, 5)


### Load the model (4-bit quantized)

In [ ]:
# Load Model with 4-bit Quantization (Lightweight)
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

model_name = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=quantization_config,
    device_map="auto",
    torch_dtype=torch.bfloat16
)

model.eval()
print("Model loaded:", model_name)


## Part 1 & Part 2 — Baseline and Best-of-N (Majority Vote)

We use a shared evaluation function from `shared/helpers.py` to run both:

- **Baseline:** `n_samples=1`, `temperature=0.0`
- **Self-consistency:** `n_samples>1`, `temperature>0.0`, with majority vote over sampled answers

This function iterates through the evaluation set, generates one or more candidate solutions per question, extracts final answers, aggregates predictions when needed, and computes accuracy and token-usage statistics.

### Part 1 — Run Baseline

The baseline is the simplest evaluation setting: the model produces one deterministic response per question, and that single extracted answer is scored directly.

Here we evaluate the model on a subset of the GSM8K evaluation set using:

- `n_samples=1`
- `temperature=0.0`

This gives us a reference point before adding extra inference-time compute through self-consistency.

In [2]:
# ## Run Baseline (System-1)
# Tip: with the 1.5B model, 100 questions should usually be much faster.

baseline_metrics, baseline_results = evaluate(
    eval_df=eval_df,
    model=model,
    tokenizer=tokenizer,
    n_samples=1,
    temperature=0.0,
    max_questions=100
)

print("Baseline metrics:")
print(json.dumps(baseline_metrics, indent=2))


NameError: name 'evaluate' is not defined

### Part 2 — Run Self-Consistency

Self-consistency increases inference-time compute by sampling multiple candidate solutions for each question and then selecting the most common final answer by majority vote.

Compared with the baseline, this approach trades additional generation cost for potentially better reasoning accuracy.

In [ ]:
# ## Run Self-Consistency (Majority Vote)
#
# If you want this to finish in a few minutes, keep n_samples modest (e.g., 5).
# Increase if you have extra time.

sc_metrics, sc_results = evaluate(
    eval_df=eval_df,
    model=model,
    tokenizer=tokenizer,
    n_samples=5,
    temperature=0.7,
    max_questions=100
)

print("Self-consistency metrics:")
print(json.dumps(sc_metrics, indent=2))


### Save results (baseline + Part 2)

In [ ]:
# ## Save Results

final_output = {
    "model_name": model_name,
    "baseline_metrics": baseline_metrics,
    "baseline_results": baseline_results,
    "self_consistency_metrics": sc_metrics,
    "self_consistency_results": sc_results,
}

with open("gsm8k_refactored_results.json", "w") as f:
    json.dump(final_output, f, indent=2)

print("Saved: gsm8k_refactored_results.json")


## Part 3 — Simple Verifier (Proposer–Verifier)

We approximate an **Outcome Reward Model (ORM)** using GSM8K labels.

- Generate **N** candidates (same as Part 2)
- Score each candidate with a **verifier** based on the true answer
- Select the candidate with the **highest verifier score**

This isolates the value of **selection** vs. **majority voting** under the same sampling budget.


In [ ]:
import re
import torch

# -----------------------------
# HARD RESET: kill old shim if it exists
# -----------------------------
globals().pop("sample_solutions", None)  # prevents the old solve_gsm8k_question-based version from being used


# -----------------------------
# Helpers
# -----------------------------
def extract_gsm8k_final_answer(x):
    """
    GSM8K label field often looks like: '... #### 42'
    Returns '42' as a string (or None).
    """
    if x is None:
        return None
    s = str(x)
    m = re.search(r"####\s*([-+]?\d[\d,]*(?:\.\d+)?)", s)
    if not m:
        return None
    return m.group(1).replace(",", "").strip()


def _extract_final_answer_from_text(text: str):
    """
    Best-effort parse model output final answer.
    1) If it contains '#### X', use that.
    2) Else use the last number that appears in the text.
    Returns a string (or None).
    """
    if text is None:
        return None
    s = str(text).strip()

    m = re.search(r"####\s*([-+]?\d[\d,]*(?:\.\d+)?)", s)
    if m:
        return m.group(1).replace(",", "").strip()

    nums = re.findall(r"[-+]?\d[\d,]*(?:\.\d+)?", s)
    if not nums:
        return None
    return nums[-1].replace(",", "").strip()


def _build_messages(question: str):
    return [
        {"role": "system", "content": "You are a helpful assistant. Solve the math problem carefully and give the final answer."},
        {"role": "user", "content": f"Solve this problem. Show your reasoning, then give the final answer.\n\nProblem: {question}\n\nFinal answer:"},
    ]


# -----------------------------
# NEW sampler (self-contained)
# -----------------------------
def sample_solutions(model, tokenizer, question, n_samples=5, temperature=0.7, max_new_tokens=256):
    """
    Generates N solutions for one question using model.generate (no solve_gsm8k_question dependency).
    Returns list of dicts: {"full_text": ..., "final_answer": ..., "gen_tokens": ...}
    """
    sols = []
    device = getattr(model, "device", "cuda" if torch.cuda.is_available() else "cpu")

    for _ in range(n_samples):
        messages = _build_messages(question)

        if hasattr(tokenizer, "apply_chat_template"):
            prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
            inputs = tokenizer(prompt, return_tensors="pt").to(device)
        else:
            prompt = messages[-1]["content"]
            inputs = tokenizer(prompt, return_tensors="pt").to(device)

        with torch.no_grad():
            out = model.generate(
                **inputs,
                do_sample=(temperature > 0),
                temperature=temperature,
                max_new_tokens=max_new_tokens,
                pad_token_id=(tokenizer.pad_token_id if tokenizer.pad_token_id is not None else tokenizer.eos_token_id),
                eos_token_id=tokenizer.eos_token_id,
            )

        gen_ids = out[0][inputs["input_ids"].shape[-1]:]
        gen_text = tokenizer.decode(gen_ids, skip_special_tokens=True)

        sols.append({
            "full_text": gen_text,
            "final_answer": _extract_final_answer_from_text(gen_text),
            "gen_tokens": int(gen_ids.shape[-1]),
        })

    return sols


# -----------------------------
# Verifier + Best-of-N (complete, no external deps)
# -----------------------------
def _to_number_maybe(x):
    if x is None:
        return None
    s = str(x).strip().replace(",", "")
    if re.fullmatch(r"[-+]?\d+\s*/\s*\d+", s):
        num, den = s.split("/")
        den = den.strip()
        if den == "0":
            return None
        return float(num) / float(den)
    try:
        if re.fullmatch(r"[-+]?\d+", s):
            return int(s)
        if re.fullmatch(r"[-+]?\d*\.\d+", s):
            return float(s)
    except Exception:
        return None
    return None


def verifier_score(pred_final_answer, true_final_answer, mode="exact"):
    pred = str(pred_final_answer).strip() if pred_final_answer is not None else ""
    true = str(true_final_answer).strip() if true_final_answer is not None else ""

    if mode == "exact":
        return 1.0 if pred == true else 0.0

    pred_num = _to_number_maybe(pred)
    true_num = _to_number_maybe(true)

    if pred_num is None or true_num is None:
        return float("-inf") if mode == "distance" else 0.0

    if mode == "numeric":
        return 1.0 if abs(float(pred_num) - float(true_num)) < 1e-9 else 0.0

    if mode == "distance":
        return -abs(float(pred_num) - float(true_num))

    raise ValueError(f"Unknown verifier mode: {mode}")


def best_of_n_with_verifier(model, tokenizer, question, true_answer, n_samples=10, temperature=0.7, max_new_tokens=256, verifier_mode="exact"):
    candidates = sample_solutions(
        model=model,
        tokenizer=tokenizer,
        question=question,
        n_samples=n_samples,
        temperature=temperature,
        max_new_tokens=max_new_tokens,
    )

    best = None
    best_score = float("-inf")
    for c in candidates:
        score = verifier_score(c.get("final_answer"), true_answer, mode=verifier_mode)
        if score > best_score:
            best_score = score
            best = c

    if best is None:
        best = candidates[0] if candidates else {"full_text": "", "final_answer": "", "gen_tokens": 0}

    best = dict(best)
    best["verifier_score"] = best_score
    best["verifier_mode"] = verifier_mode
    best["n_samples"] = n_samples
    best["all_candidates"] = candidates  # TTC accounting
    return best


def evaluate_best_of_n_verifier(
    eval_df,
    model,
    tokenizer,
    n_samples=10,
    temperature=0.7,
    max_new_tokens=256,
    verifier_mode="exact",
):
    rows = []
    correct = 0
    total_tokens = 0

    # 🔹 Progress bar over questions
    for i, row in tqdm(
        eval_df.iterrows(),
        total=len(eval_df),
        desc=f"Best-of-{n_samples} + verifier",
    ):
        q = row["question"]
        true = extract_gsm8k_final_answer(row["answer"])

        best = best_of_n_with_verifier(
            model=model,
            tokenizer=tokenizer,
            question=q,
            true_answer=true,
            n_samples=n_samples,
            temperature=temperature,
            max_new_tokens=max_new_tokens,
            verifier_mode=verifier_mode,
        )

        pred = best.get("final_answer")

        if verifier_mode == "exact":
            is_correct = (str(pred).strip() == str(true).strip())
        else:
            is_correct = (verifier_score(pred, true, mode="numeric") == 1.0)

        correct += int(is_correct)
        total_tokens += sum(c.get("gen_tokens", 0) for c in best["all_candidates"])

        rows.append({
            "idx": int(i),
            "question": q,
            "true_answer": true,
            "pred_answer": pred,
            "correct": bool(is_correct),
            "verifier_score": best.get("verifier_score"),
        })

    metrics = {
        "method": "best_of_n_verifier",
        "n_samples": n_samples,
        "temperature": temperature,
        "max_new_tokens": max_new_tokens,
        "verifier_mode": verifier_mode,
        "accuracy": correct / max(1, len(eval_df)),
        "avg_tokens": total_tokens / max(1, len(eval_df)),
    }

    return metrics, rows






### Run Part 3 (Best-of-N + verifier)
Pick one N (e.g., 5 or 10) to start, then scale.

In [ ]:
# -----------------------------
# Example run
# -----------------------------
bover_metrics, bover_rows = evaluate_best_of_n_verifier(
    eval_df=eval_df.head(5),
    model=model,
    tokenizer=tokenizer,
    n_samples=10,
    temperature=0.7,
    max_new_tokens=256,
    verifier_mode="exact",
)
print(bover_metrics)

## Part 4 — Difficulty Binning (Easy/Medium/Hard)

We estimate empirical difficulty by running the baseline solver **K times** with mild randomness and computing a pass rate.

Bins:
- **Easy:** pass rate ≥ 0.8  
- **Medium:** 0.3 ≤ pass rate < 0.8  
- **Hard:** pass rate < 0.3

We then report method performance broken down by difficulty.


In [ ]:
import pandas as pd

# =========================
# Part 4 — Difficulty Bins
# =========================

def assign_difficulty_bin(pass_rate):
    if pass_rate >= 0.8:
        return "easy"
    if pass_rate >= 0.3:
        return "medium"
    return "hard"


def estimate_pass_rate(model, tokenizer, question, true_answer, k=5, temperature=0.3, max_new_tokens=256):
    """
    Estimate pass@k using sample_solutions(...) (no solve_gsm8k_question dependency).
    Returns: (pass_rate, total_tokens_used)
    """ Oh you got your Christmas letter with
    sols = sample_solutions(
        model=model,
        tokenizer=tokenizer,
        question=question,
        n_samples=k,
        temperature=temperature,
        max_new_tokens=max_new_tokens,
    )

    correct = 0
    total_tokens = 0
    for s in sols:
        pred = s.get("final_answer")
        total_tokens += int(s.get("gen_tokens", 0))
        if str(pred).strip() == str(true_answer).strip():
            correct += 1

    return correct / max(1, k), total_tokens

def build_difficulty_table(df, model, tokenizer, k=5, temperature=0.3, max_new_tokens=256, max_items=300):
    out = []
    use_df = df.head(max_items) if max_items is not None else df

    # 🔹 Progress bar over questions
    for i, row in tqdm(
        use_df.iterrows(),
        total=len(use_df),
        desc=f"Difficulty bins (k={k})"
    ):
        q = row["question"]

        # GSM8K: extract numeric answer from raw "answer" field
        true = extract_gsm8k_final_answer(row["answer"])

        pr, toks = estimate_pass_rate(
            model=model,
            tokenizer=tokenizer,
            question=q,
            true_answer=true,
            k=k,
            temperature=temperature,
            max_new_tokens=max_new_tokens,
        )

        out.append({
            "idx": int(i),
            "pass_rate": pr,
            "difficulty": assign_difficulty_bin(pr),
            "tokens_used_for_estimate": toks,
        })

    return pd.DataFrame(out)


# Build difficulty labels on train_subset (use fewer for speed, then scale)
difficulty_train = build_difficulty_table(
    train_df, model, tokenizer,
    k=5, temperature=0.3, max_new_tokens=256,
    max_items=5  # ✅ start with 5 to confirm it works; raise later
)

print(difficulty_train["difficulty"].value_counts())
difficulty_train


### Attach difficulty to eval subset and compute per-bin metrics

## Part 5 — Sequential Revisions (Self-Refinement)

We create a revision chain:
- Attempt 0: baseline solution
- Attempts 1..R: revise the previous attempt

Selection options:
- **Verifier selection**: choose the attempt with the best verifier score
- **Majority vote** over attempt answers

We compare revisions against Best-of-N under a similar generation budget.


In [ ]:
# 1) make eval_with_diff have an idx column
eval_with_diff = eval_df.reset_index().rename(columns={"index": "idx"})

# 2) compute difficulty for a small eval subset (5 items)
difficulty_eval = build_difficulty_table(
    eval_with_diff,
    model,
    tokenizer,
    k=5,
    temperature=0.3,
    max_new_tokens=128,
    max_items=5
)

# 3) merge difficulty labels back onto eval_with_diff
eval_with_diff = eval_with_diff.merge(
    difficulty_eval[["idx", "difficulty", "pass_rate"]],
    on="idx",
    how="left"
)

eval_with_diff[["idx", "difficulty", "pass_rate"]].head(10)


In [ ]:
from tqdm.auto import tqdm

# =========================
# Part 5 — Revisions (NO solve_gsm8k_question)
# =========================

def make_revision_prompt(question, previous_attempt_text):
    return (
        "You previously attempted this question and produced the following reasoning and answer.\n"
        "Your reasoning may contain mistakes.\n"
        "Carefully analyze it, explain what is wrong, and then write a corrected solution.\n"
        "At the end, output on a new line exactly: Final Answer: <number>\n\n"
        f"Question:\n{question}\n\n"
        f"Previous Attempt:\n{previous_attempt_text}\n"
    )


def extract_final_answer_revision(text: str):
    """
    For revision outputs, we request: 'Final Answer: <number>'.
    Fallbacks to your general answer extraction if needed.
    """
    if text is None:
        return None
    s = str(text)

    m = re.search(r"Final Answer:\s*([-+]?\d[\d,]*(?:\.\d+)?)", s)
    if m:
        return m.group(1).replace(",", "").strip()

    # fallback: reuse your generic extractor if present
    return _extract_final_answer_from_text(s)  # you already defined this earlier


def generate_with_prompt(model, tokenizer, prompt_text, temperature=0.7, max_new_tokens=256):
    messages = [{"role": "user", "content": prompt_text}]
    device = getattr(model, "device", "cuda" if torch.cuda.is_available() else "cpu")

    if hasattr(tokenizer, "apply_chat_template"):
        prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer(prompt, return_tensors="pt").to(device)
    else:
        inputs = tokenizer(prompt_text, return_tensors="pt").to(device)

    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            do_sample=(temperature > 0),
            pad_token_id=(tokenizer.pad_token_id if tokenizer.pad_token_id is not None else tokenizer.eos_token_id),
            eos_token_id=tokenizer.eos_token_id,
        )

    gen_ids = out[0][inputs["input_ids"].shape[-1]:]
    gen_text = tokenizer.decode(gen_ids, skip_special_tokens=True)
    gen_tokens = int(gen_ids.shape[-1])

    final_ans = extract_final_answer_revision(gen_text)
    return gen_text, final_ans, gen_tokens


def run_revision_chain(
    model, tokenizer, question,
    r_steps=3,
    base_temperature=0.0,
    rev_temperature=0.7,
    max_new_tokens=256
):
    attempts = []

    # Attempt 0: baseline attempt (single sample)
    a0 = sample_solutions(
        model=model,
        tokenizer=tokenizer,
        question=question,
        n_samples=1,
        temperature=base_temperature,
        max_new_tokens=max_new_tokens
    )[0]
    a0_text = a0.get("full_text", "")
    a0_ans  = a0.get("final_answer", None)
    a0_toks = int(a0.get("gen_tokens", 0))

    attempts.append({"attempt": 0, "full_text": a0_text, "final_answer": a0_ans, "gen_tokens": a0_toks})

    prev_text = a0_text
    for k in range(r_steps):
        rp = make_revision_prompt(question, prev_text)
        txt, ans, toks = generate_with_prompt(
            model=model,
            tokenizer=tokenizer,
            prompt_text=rp,
            temperature=rev_temperature,
            max_new_tokens=max_new_tokens,
        )
        attempts.append({"attempt": k + 1, "full_text": txt, "final_answer": ans, "gen_tokens": toks})
        prev_text = txt

    return attempts


def select_from_attempts(attempts, true_answer=None, selection="verifier", verifier_mode="exact"):
    if selection == "last":
        out = dict(attempts[-1])
        out["selection"] = "last"
        return out

    if selection == "majority":
        answers = [str(a.get("final_answer", "")).strip() for a in attempts]
        counts = {}
        for a in answers:
            counts[a] = counts.get(a, 0) + 1
        best_answer = sorted(counts.items(), key=lambda x: (-x[1], x[0]))[0][0]
        for a in attempts:
            if str(a.get("final_answer", "")).strip() == best_answer:
                out = dict(a)
                out["selection"] = "majority"
                return out

    if selection == "verifier":
        if true_answer is None:
            raise ValueError("true_answer is required for verifier selection.")
        best = None
        best_score = float("-inf")
        for a in attempts:
            s = verifier_score(a.get("final_answer", ""), true_answer, mode=verifier_mode)
            if s > best_score:
                best_score = s
                best = a
        out = dict(best)
        out["selection"] = "verifier"
        out["verifier_score"] = best_score
        return out

    raise ValueError(f"Unknown selection: {selection}")


def evaluate_revisions(
    eval_df,
    model,
    tokenizer,
    r_steps=3,
    base_temperature=0.0,
    rev_temperature=0.7,
    max_new_tokens=256,
    selection="verifier",
    verifier_mode="exact",
):
    rows = []
    correct = 0
    total_tokens = 0

    for i, row in tqdm(eval_df.iterrows(), total=len(eval_df), desc=f"Revisions (r_steps={r_steps})"):
        q = row["question"]
        true = extract_gsm8k_final_answer(row["answer"])  # ✅ GSM8K label extraction

        chain = run_revision_chain(
            model=model,
            tokenizer=tokenizer,
            question=q,
            r_steps=r_steps,
            base_temperature=base_temperature,
            rev_temperature=rev_temperature,
            max_new_tokens=max_new_tokens,
        )

        chosen = select_from_attempts(chain, true_answer=true, selection=selection, verifier_mode=verifier_mode)

        pred = chosen.get("final_answer", "")
        is_correct = (str(pred).strip() == str(true).strip())

        correct += int(is_correct)
        chain_tokens = int(sum(a.get("gen_tokens", 0) for a in chain))
        total_tokens += chain_tokens

        rows.append({
            "idx": int(row["idx"]) if "idx" in row else int(i),
            "difficulty": row.get("difficulty", None),
            "question": q,
            "true_answer": true,
            "pred_answer": pred,
            "correct": bool(is_correct),
            "selection": selection,
            "total_chain_tokens": chain_tokens,
            "attempts": chain,
        })

    metrics = {
        "method": "revisions",
        "r_steps": r_steps,
        "base_temperature": base_temperature,
        "rev_temperature": rev_temperature,
        "max_new_tokens": max_new_tokens,
        "selection": selection,
        "verifier_mode": verifier_mode,
        "accuracy": correct / max(1, len(eval_df)),
        "avg_tokens": total_tokens / max(1, len(eval_df)),
    }
    return metrics, rows


### Run Part 5 (Revisions)
Start with 50–100 examples; revisions are slower.

In [ ]:
rev_metrics, rev_rows = evaluate_revisions(
    eval_df=eval_with_diff.head(5),
    model=model,
    tokenizer=tokenizer,
    r_steps=3,
    base_temperature=0.0,
    rev_temperature=0.7,
    max_new_tokens=256,
    selection="verifier",
    verifier_mode="exact"
)
print(rev_metrics)


## Part 6 — Lightweight Beam/Search

This is a **simple, teaching-friendly** beam-style search over *partial reasoning*:

- Keep a beam of **B** partial solutions
- Expand each partial solution with **M** candidate “next steps”
- Score partials using a heuristic (here: **average logprob per generated token** with a length penalty)
- Stop if a candidate emits a `Final Answer: ...` or after `max_steps`

Then select the best finished candidate using the verifier.

Because it’s compute-heavy, run on **100–200** examples.


In [ ]:
import re
import torch
from tqdm.auto import tqdm

# =========================
# Helper: Final Answer Extraction
# =========================

def extract_final_answer(text: str):
    """
    Beam / step prompts request:
      'Final Answer: <number>'
    Fallbacks to generic numeric extraction.
    """
    if text is None:
        return None
    s = str(text)

    m = re.search(r"Final Answer:\s*([-+]?\d[\d,]*(?:\.\d+)?)", s)
    if m:
        return m.group(1).replace(",", "").strip()

    return _extract_final_answer_from_text(s)


# =========================
# Beam Search Utilities
# =========================

def generate_continuations_with_scores(
    model,
    tokenizer,
    prompt_text,
    num_return_sequences=2,
    temperature=0.7,
    max_new_tokens=80
):
    """
    Generate multiple continuations and compute a heuristic avg log-prob score.
    Handles tokenizers where apply_chat_template returns either:
      - a dict-like BatchEncoding, or
      - a plain Tensor of input_ids
    """
    messages = [{"role": "user", "content": prompt_text}]
    device = getattr(model, "device", "cuda" if torch.cuda.is_available() else "cpu")

    # --- Robust encoding across tokenizer variants ---
    if hasattr(tokenizer, "apply_chat_template"):
        enc = tokenizer.apply_chat_template(
            messages,
            tokenize=True,
            return_tensors="pt"
        )
        if isinstance(enc, torch.Tensor):
            # Some tokenizers return input_ids tensor directly
            input_ids = enc.to(device)
            attention_mask = torch.ones_like(input_ids, device=device)
            enc = {"input_ids": input_ids, "attention_mask": attention_mask}
        else:
            # BatchEncoding / dict-like
            enc = {k: v.to(device) for k, v in enc.items()}
            if "attention_mask" not in enc:
                enc["attention_mask"] = torch.ones_like(enc["input_ids"], device=device)
    else:
        # Fallback: plain tokenization
        enc = tokenizer(prompt_text, return_tensors="pt")
        enc = {k: v.to(device) for k, v in enc.items()}
        if "attention_mask" not in enc:
            enc["attention_mask"] = torch.ones_like(enc["input_ids"], device=device)

    out = model.generate(
        **enc,
        max_new_tokens=max_new_tokens,
        do_sample=(temperature > 0),
        temperature=temperature,
        num_return_sequences=num_return_sequences,
        return_dict_in_generate=True,
        output_scores=True,
        pad_token_id=(tokenizer.pad_token_id if tokenizer.pad_token_id is not None else tokenizer.eos_token_id),
        eos_token_id=tokenizer.eos_token_id,
    )

    sequences = out.sequences
    scores_per_step = out.scores  # list[T] of (batch, vocab)

    results = []
    prompt_len = enc["input_ids"].shape[-1]

    for b in range(sequences.shape[0]):
        full_ids = sequences[b]
        gen_ids = full_ids[prompt_len:]
        gen_tokens = int(gen_ids.shape[-1])
        text = tokenizer.decode(full_ids, skip_special_tokens=True)

        avg_logprob = 0.0
        if gen_tokens > 0:
            logps = []
            for t in range(gen_tokens):
                step_scores = scores_per_step[t][b]
                step_logp = torch.log_softmax(step_scores, dim=-1)
                tok_id = int(gen_ids[t].item())
                logps.append(float(step_logp[tok_id].item()))
            avg_logprob = float(sum(logps) / max(1, len(logps)))

        results.append({
            "text": text,
            "gen_tokens": gen_tokens,
            "avg_logprob": avg_logprob,
            "final_answer": extract_final_answer(text),
        })

    return results



def make_step_prompt(question, partial_reasoning):
    return (
        "You are solving a math word problem.\n"
        "Extend the solution by ONE clear next step.\n"
        "If you are ready to finish, output on a new line exactly: Final Answer: <number>\n\n"
        f"Question:\n{question}\n\n"
        f"Current Partial Solution:\n{partial_reasoning}\n"
    )


# =========================
# Beam Search Core
# =========================

def beam_search_gsm8k(
    model,
    tokenizer,
    question,
    true_answer=None,
    beam_size=3,
    expansions_per_beam=2,
    max_steps=6,
    temperature=0.7,
    step_max_new_tokens=80,
    length_penalty=0.02,
    verifier_mode="exact",
):
    beam = [{
        "partial": "",
        "score": 0.0,
        "steps": 0,
        "tokens": 0,
        "finished": False,
        "final_answer": None,
    }]

    finished = []

    for step in range(max_steps):
        new_candidates = []

        for b in beam:
            if b["finished"]:
                new_candidates.append(b)
                continue

            prompt = make_step_prompt(question, b["partial"])
            conts = generate_continuations_with_scores(
                model=model,
                tokenizer=tokenizer,
                prompt_text=prompt,
                num_return_sequences=expansions_per_beam,
                temperature=temperature,
                max_new_tokens=step_max_new_tokens,
            )

            for c in conts:
                new_partial = c["text"]
                total_tokens = b["tokens"] + c["gen_tokens"]

                score = (
                    b["score"]
                    + c["avg_logprob"]
                    - (length_penalty * total_tokens)
                )

                is_finished = (
                    c["final_answer"] is not None
                    and str(c["final_answer"]).strip() != ""
                )

                new_candidates.append({
                    "partial": new_partial,
                    "score": score,
                    "steps": step + 1,
                    "tokens": total_tokens,
                    "finished": is_finished,
                    "final_answer": c["final_answer"],
                })

        beam = sorted(new_candidates, key=lambda x: x["score"], reverse=True)[:beam_size]

        for b in beam:
            if b["finished"]:
                finished.append(b)

        if len(finished) >= beam_size:
            break

    pool = finished if finished else beam

    if true_answer is not None:
        best = None
        best_vs = float("-inf")
        for p in pool:
            vs = verifier_score(p.get("final_answer", ""), true_answer, mode=verifier_mode)
            if vs > best_vs:
                best_vs = vs
                best = p
        best = dict(best)
        best["verifier_score"] = best_vs
        best["verifier_mode"] = verifier_mode
        best["used_pool"] = "finished" if finished else "beam"
        return best

    return pool[0]


# =========================
# Evaluation Wrapper
# =========================

def evaluate_beam_search(
    eval_df,
    model,
    tokenizer,
    beam_size=3,
    expansions_per_beam=2,
    max_steps=6,
    temperature=0.7,
    step_max_new_tokens=80,
    length_penalty=0.02,
    verifier_mode="exact",
):
    rows = []
    correct = 0
    total_tokens = 0

    for i, row in tqdm(
        eval_df.iterrows(),
        total=len(eval_df),
        desc=f"Beam search (B={beam_size})",
    ):
        q = row["question"]
        true = extract_gsm8k_final_answer(row["answer"])

        best = beam_search_gsm8k(
            model=model,
            tokenizer=tokenizer,
            question=q,
            true_answer=true,
            beam_size=beam_size,
            expansions_per_beam=expansions_per_beam,
            max_steps=max_steps,
            temperature=temperature,
            step_max_new_tokens=step_max_new_tokens,
            length_penalty=length_penalty,
            verifier_mode=verifier_mode,
        )

        pred = best.get("final_answer", "")
        is_correct = (str(pred).strip() == str(true).strip())

        correct += int(is_correct)
        total_tokens += int(best.get("tokens", 0))

        rows.append({
            "idx": int(row["idx"]) if "idx" in row else int(i),
            "difficulty": row.get("difficulty", None),
            "question": q,
            "true_answer": true,
            "pred_answer": pred,
            "correct": bool(is_correct),
            "tokens": int(best.get("tokens", 0)),
            "score": float(best.get("score", 0.0)),
            "verifier_score": best.get("verifier_score", None),
            "text": best.get("partial", ""),
        })

    metrics = {
        "method": "beam_search",
        "beam_size": beam_size,
        "expansions_per_beam": expansions_per_beam,
        "max_steps": max_steps,
        "temperature": temperature,
        "step_max_new_tokens": step_max_new_tokens,
        "length_penalty": length_penalty,
        "verifier_mode": verifier_mode,
        "accuracy": correct / max(1, len(eval_df)),
        "avg_tokens": total_tokens / max(1, len(eval_df)),
    }

    return metrics, rows


### Run Part 6 (Beam/Search)
Run on ~100 examples first (this can be slower than revisions).

In [ ]:
beam_metrics, beam_rows = evaluate_beam_search(
    eval_df=eval_with_diff.head(5),  # start small
    model=model,
    tokenizer=tokenizer,
    beam_size=3,
    expansions_per_beam=2,
    max_steps=6,
    temperature=0.7,
    step_max_new_tokens=80,
    length_penalty=0.02,
    verifier_mode="exact",
)

print(beam_metrics)


## Part 7 — Difficulty-Aware, Compute-Optimal Policy

We route questions by difficulty bin to different TTC strategies:

- **S1:** Baseline (single pass)
- **S2:** Best-of-N (small N) majority
- **S3:** Revisions (small R) + verifier selection
- **S4:** High compute (larger N or beam search)

You will compare policies like A/B/C against uniform strategies (always S2, always S3, etc.).


In [ ]:
from tqdm.auto import tqdm

# =========================
# Part 7 — Routing Policies
# =========================

def run_strategy(row, model, tokenizer, strategy_name, cfg):
    q = row["question"]
    true = row["answer"]

    if strategy_name == "S1":
        # If you don't have solve_gsm8k_question anymore, swap this to sample_solutions(..., n_samples=1)
        full_text, pred, toks = solve_gsm8k_question(
            model=model, tokenizer=tokenizer, question=q,
            temperature=cfg.get("temperature", 0.0),
            max_new_tokens=cfg.get("max_new_tokens", 256)
        )
        return {"pred": pred, "tokens": toks, "text": full_text}

    if strategy_name == "S2":
        sols = sample_solutions(
            model=model, tokenizer=tokenizer, question=q,
            n_samples=cfg.get("n_samples", 5),
            temperature=cfg.get("temperature", 0.7),
            max_new_tokens=cfg.get("max_new_tokens", 256)
        )
        answers = [str(s.get("final_answer","")).strip() for s in sols]
        counts = {}
        for a in answers:
            counts[a] = counts.get(a, 0) + 1
        best_answer = sorted(counts.items(), key=lambda x: (-x[1], x[0]))[0][0]

        chosen = None
        for s in sols:
            if str(s.get("final_answer","")).strip() == best_answer:
                chosen = s
                break

        total_tokens = sum([s.get("gen_tokens",0) for s in sols])
        return {
            "pred": chosen.get("final_answer","") if chosen else best_answer,
            "tokens": total_tokens,
            "text": chosen.get("full_text","") if chosen else ""
        }

    if strategy_name == "S3":
        chain = run_revision_chain(
            model=model, tokenizer=tokenizer, question=q,
            r_steps=cfg.get("r_steps", 2),
            base_temperature=cfg.get("base_temperature", 0.0),
            rev_temperature=cfg.get("rev_temperature", 0.7),
            max_new_tokens=cfg.get("max_new_tokens", 256)
        )
        chosen = select_from_attempts(
            chain, true_answer=true,
            selection=cfg.get("selection", "verifier"),
            verifier_mode=cfg.get("verifier_mode", "exact")
        )
        total_tokens = sum([a.get("gen_tokens",0) for a in chain])
        return {"pred": chosen.get("final_answer",""), "tokens": total_tokens, "text": chosen.get("full_text","")}

    if strategy_name == "S4":
        if cfg.get("mode","beam") == "beam":
            best = beam_search_gsm8k(
                model=model, tokenizer=tokenizer, question=q, true_answer=true,
                beam_size=cfg.get("beam_size", 3),
                expansions_per_beam=cfg.get("expansions_per_beam", 2),
                max_steps=cfg.get("max_steps", 6),
                temperature=cfg.get("temperature", 0.7),
                step_max_new_tokens=cfg.get("step_max_new_tokens", 80),
                length_penalty=cfg.get("length_penalty", 0.02),
                verifier_mode=cfg.get("verifier_mode", "exact")
            )
            return {"pred": best.get("final_answer",""), "tokens": int(best.get("tokens",0)), "text": best.get("partial","")}
        else:
            best = best_of_n_with_verifier(
                model=model, tokenizer=tokenizer, question=q, true_answer=true,
                n_samples=cfg.get("n_samples", 20),
                temperature=cfg.get("temperature", 0.7),
                max_new_tokens=cfg.get("max_new_tokens", 256),
                verifier_mode=cfg.get("verifier_mode", "exact")
            )
            # NOTE: best_of_n_with_verifier returns best candidate; token accounting can vary by your implementation
            return {"pred": best.get("final_answer",""), "tokens": best.get("gen_tokens",0), "text": best.get("full_text","")}

    raise ValueError(f"Unknown strategy: {strategy_name}")


def evaluate_policy(eval_df, model, tokenizer, policy_map, cfg_map, desc="Policy"):
    rows = []
    correct = 0
    total_tokens = 0

    for i, row in tqdm(eval_df.iterrows(), total=len(eval_df), desc=desc):
        diff = row.get("difficulty", "medium")
        strat = policy_map.get(diff, "S2")
        out = run_strategy(row, model, tokenizer, strat, cfg_map.get(strat, {}))

        pred = out["pred"]
        true = row["answer"]
        is_correct = (str(pred).strip() == str(true).strip())

        correct += int(is_correct)
        total_tokens += int(out["tokens"])

        rows.append({
            "idx": int(i),
            "difficulty": diff,
            "strategy": strat,
            "answer": true,
            "pred_answer": pred,
            "correct": bool(is_correct),
            "tokens": int(out["tokens"])
        })

    acc = correct / max(1, len(eval_df))
    avg_tokens = total_tokens / max(1, len(eval_df))
    return {"accuracy": acc, "avg_tokens": avg_tokens}, rows


# Example policy definitions (edit as desired)
policy_A = {"easy": "S1", "medium": "S2", "hard": "S3"}
policy_B = {"easy": "S1", "medium": "S3", "hard": "S4"}
policy_C = {"easy": "S2", "medium": "S3", "hard": "S4"}

cfg_map = {
    "S1": {"temperature": 0.0, "max_new_tokens": 256},
    "S2": {"n_samples": 5, "temperature": 0.7, "max_new_tokens": 256},
    "S3": {"r_steps": 2, "base_temperature": 0.0, "rev_temperature": 0.7, "max_new_tokens": 256, "selection":"verifier", "verifier_mode":"exact"},
    "S4": {"mode":"beam", "beam_size": 3, "expansions_per_beam": 2, "max_steps": 6, "temperature": 0.7, "step_max_new_tokens": 80, "length_penalty": 0.02, "verifier_mode":"exact"}
}

# Evaluate on a subset first (with progress bars)
subset = eval_with_diff.head(120)

polA_metrics, polA_rows = evaluate_policy(subset, model, tokenizer, policy_A, cfg_map, desc="Policy A")
polB_metrics, polB_rows = evaluate_policy(subset, model, tokenizer, policy_B, cfg_map, desc="Policy B")
polC_metrics, polC_rows = evaluate_policy(subset, model, tokenizer, policy_C, cfg_map, desc="Policy C")

print("Policy A:", polA_metrics)
print("Policy B:", polB_metrics)
print("Policy C:", polC_metrics)

## Results Summary + Plots

Fill this section with your final runs (200–500 eval questions).  
Minimum plots/tables to include in your report:

- Accuracy vs. Avg tokens for:
  - Baseline (Part 1)
  - Best-of-N majority (Part 2)
  - Best-of-N verifier (Part 3)
  - Revisions (Part 5)
  - Beam/search (Part 6; on a subset is fine)
- Accuracy by difficulty bin (Easy/Medium/Hard)
- Policy routing comparison (Part 7)


In [ ]:
# Collect the metrics you ran above into one DataFrame for quick comparison.
metric_rows = []

# These variables exist if you ran the example cells; add/remove as needed.
for m in [baseline_metrics, sc_metrics, bover_metrics, rev_metrics, beam_metrics]:
    if isinstance(m, dict):
        metric_rows.append(m)

metrics_df = pd.DataFrame(metric_rows)
metrics_df


In [ ]:
# Simple plot: accuracy vs avg_tokens
if "accuracy" in metrics_df.columns and "avg_tokens" in metrics_df.columns:
    plt.figure()
    plt.scatter(metrics_df["avg_tokens"], metrics_df["accuracy"])
    for _, r in metrics_df.iterrows():
        label = r.get("method","")
        plt.annotate(label, (r["avg_tokens"], r["accuracy"]))
    plt.xlabel("Avg tokens (TTC proxy)")
    plt.ylabel("Accuracy")
    plt.title("Accuracy vs Test-Time Compute (tokens)")
    plt.show()


## Report Template (3–5 pages)

Use this structure (you can paste into a Google Doc or write directly here):

1. **Motivation**
   - What is test-time compute (TTC)?
   - Why GSM8K and why a lightweight open model?
   - What you expect: sampling, selection, revisions, and search trade compute for accuracy.

2. **Methods**
   - Dataset split sizes and preprocessing
   - Model details (Qwen2.5-1.5B, quantized), decoding settings
   - Implemented strategies (Parts 1–7), with key hyperparameters (N, R, B, M, etc.)
   - Compute metric: average generated tokens (and/or wall-clock, if recorded)

3. **Results**
   - Overall accuracy vs compute tradeoff for each method
   - Breakdown by difficulty bin (Easy/Medium/Hard)
   - Policy routing comparison vs uniform strategies

4. **Discussion**
   - Where did TTC help most (which difficulty bins)?
   - Majority vote vs verifier selection: when does each win?
   - Revisions vs Best-of-N under similar compute budgets
   - Is beam/search “worth it” for this model/budget?
   - Connection back to the behaviors described in Snell-style TTC work

5. **Limitations**
   - Token proxy vs true FLOPs/time
   - Heuristic scoring for beam search (no learned PRM)
   - Small model constraints and prompt sensitivity


In [ ]:
# Optional: export your metrics and selected rows to JSON for your report
out = {
    "metrics_table": metrics_df.to_dict(orient="records"),
    "policy_A": polA_metrics,
    "policy_B": polB_metrics,
    "policy_C": polC_metrics,
}
with open("hw_results_summary.json","w") as f:
    json.dump(out, f, indent=2)
print("Wrote hw_results_summary.json")


In [ ]:
NOTEBOOK_END_TIME = time.time()
total_runtime = NOTEBOOK_END_TIME - NOTEBOOK_START_TIME

print("\n===== TOTAL NOTEBOOK RUNTIME =====")
print("Total time (seconds):", round(total_runtime, 2))
print("Total time (minutes):", round(total_runtime / 60, 2))